# Set up (run it only once)

In [ ]:
!python -m venv .venv # creates the environment 

Please activate the environment before to proceed

In [ ]:
%pip install scanpy anndata pandas scipy openpyxl pydeseq2

In [ ]:
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
import os
import numpy as np
import matplotlib.pyplot as plt
import gc


# Single cell preprocessing pipeline

## Helper functions

In [ ]:
def read_count_matrix(file_path, sample_name):
    # The read_count_matrix function is taking a file path and the sample name as input and returns the count matrix.

    counts = pd.read_csv(file_path, sep="\t", index_col=0)

    counts.index = counts.index.astype(str).str.replace('"', '', regex=False) # getting all gene names
    counts.columns = counts.columns.astype(str).str.replace('"', '', regex=False)  # getting all cell tags

    # Convert into the format required by Scanpy/Anndata: cells × genes (= transposing)
    counts = counts.T  # cells × genes (transpose)

    adata = ad.AnnData(
        X=sparse.csr_matrix(counts.values), # transcription matrix
        obs=pd.DataFrame(index=counts.index), # cells are treated as observations
        var=pd.DataFrame(index=counts.columns) # genes are treated as variables 
    )

    adata.obs["sample"] = sample_name
    adata.layers["counts"] = adata.X.copy()
    adata.var_names_make_unique() # makes gene names unique (e.g. multiple transcripts for the same gene)

    return adata

In [ ]:
def remove_nan_genes(adata):
    # QC step: Remove genes that are not expressed
    # Convert X to dense array if needed
    X = adata.X.toarray() if sparse.issparse(adata.X) else adata.X
    
    # Keep genes without NaN or infinite values
    keep_genes = np.isfinite(X).all(axis=0)
    
    print("Genes before removing NaN genes:", adata.n_vars)
    print("Genes after removing NaN genes:", keep_genes.sum())
    
    return adata[:, keep_genes].copy()

In [ ]:
def de_disease_group(adata, disease_name, de_layer="combat"):
    # performing a differential analysis based on disease phenotype (e.g. FSHD1, FSHD2 compared to controls)
    sub = adata[
        (adata.obs["disease_group"] == disease_name) &
        (adata.obs["cell_classification"].isin([
            "DUX4-affected",
            "Non-affected late myocytes"
        ]))
    ].copy()
    
    # Remove unused categories
    sub.obs["cell_classification"] = sub.obs["cell_classification"].astype("category")
    sub.obs["cell_classification"] = sub.obs["cell_classification"].cat.remove_unused_categories()
    
    # Count cells in each group
    group_counts = sub.obs["cell_classification"].value_counts()
    
    print("\nSample:", disease_name)
    print(group_counts)
    
    
    # Check minimum cell number
    if group_counts["DUX4-affected"] < 2:
        print("Skipped: DUX4-affected group has fewer than 2 cells.")
        return None
    
    if group_counts["Non-affected late myocytes"] < 2:
        print("Skipped: reference group has fewer than 2 cells.")
        return None
    
    # Run Wilcoxon differential expression analysis
    sc.tl.rank_genes_groups(
        sub,
        groupby="cell_classification",
        groups=["DUX4-affected"],
        reference="Non-affected late myocytes",
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        use_raw=False
    )
    
    # Convert result to dataframe
    de_df = sc.get.rank_genes_groups_df(
        sub,
        group="DUX4-affected"
    )
    
    de_df = de_df.rename(columns={"names": "gene_id"})
    de_df["disease_group"] = disease_name
    
    return de_df

In [ ]:
def run_de_for_sample(adata, sample_name):
    sub = adata[
        (adata.obs["sample_batch"] == sample_name) &
        (adata.obs["cell_classification"].isin([
            "DUX4-affected",
            "Non-affected late myocytes"
        ]))
    ].copy()
    
    # Remove unused categories
    sub.obs["cell_classification"] = sub.obs["cell_classification"].astype("category")
    sub.obs["cell_classification"] = sub.obs["cell_classification"].cat.remove_unused_categories()
    
    # Count cells in each group
    group_counts = sub.obs["cell_classification"].value_counts()
    
    print("\nSample:", sample_name)
    print(group_counts)
    
    
    # Check minimum cell number
    if group_counts["DUX4-affected"] < 2:
        print("Skipped: DUX4-affected group has fewer than 2 cells.")
        return None
    
    if group_counts["Non-affected late myocytes"] < 2:
        print("Skipped: reference group has fewer than 2 cells.")
        return None
    
    # Run Wilcoxon differential expression analysis
    sc.tl.rank_genes_groups(
        sub,
        groupby="cell_classification",
        groups=["DUX4-affected"],
        reference="Non-affected late myocytes",
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        use_raw=False
    )
    
    # Convert result to dataframe
    de_df = sc.get.rank_genes_groups_df(
        sub,
        group="DUX4-affected"
    )
    
    de_df = de_df.rename(columns={"names": "gene_id"})
    de_df["sample"] = sample_name
    
    return de_df


In [ ]:
# GFF3 standard column names
gff_cols = [
    "seqid", "source", "type", "start", "end",
    "score", "strand", "phase", "attributes"
]

# Read GFF3 file and skip comment lines
gff = pd.read_csv(
    file_path_annotation,
    sep="\t",
    comment="#",
    header=None,
    names=gff_cols,
    engine="python"
)

# Keep only gene-level annotation
genes = gff[gff["type"] == "gene"].copy()

# Extract Ensembl gene ID and gene symbol
genes["gene_id"] = genes["attributes"].str.extract(r"(?:^|;)gene_id=([^;]+)")
genes["gene_name"] = genes["attributes"].str.extract(r"(?:^|;)gene_name=([^;]+)")

# Clean Ensembl ID: remove version number
genes["gene_id_clean"] = (
    genes["gene_id"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.\d+$", "", regex=True)
)

# Build annotation table
annotation = (
    genes[["gene_id_clean", "gene_name"]]
    .dropna()
    .drop_duplicates()
)

# Create mapping dictionary: clean Ensembl ID -> gene symbol
id_to_symbol = dict(zip(annotation["gene_id_clean"], annotation["gene_name"]))


In [ ]:
def annotate_gene_set(gene_set, id_to_symbol, gene_set_name):
    df = pd.DataFrame({
        "ensembl_id": sorted(list(gene_set))
    })
    
    df["ensembl_id_clean"] = (
        df["ensembl_id"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.\d+$", "", regex=True)
    )
    
    df["gene_symbol"] = df["ensembl_id_clean"].map(id_to_symbol)
    df.insert(0, "gene_set", gene_set_name)
    
    return df

In [ ]:
# function for volcano plot
def plot_volcano(
    de_df,
    title,
    gene_col="gene_symbol",
    logfc_col="logfoldchanges",
    padj_col="pvals_adj",
    padj_cutoff=0.05,
    logfc_cutoff=1,
    label_top=10     # label the top 10 expressed genes
):
    df = de_df.copy()
    
    # Remove missing or infinite values
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=[logfc_col, padj_col])
    
    # Avoid -log10(0)
    df[padj_col] = df[padj_col].replace(0, 1e-300)
    
    # Calculate -log10 adjusted p-value
    df["minus_log10_padj"] = -np.log10(df[padj_col])
    
    # Define significant genes
    if logfc_cutoff is None:
        df["significant"] = df[padj_col] < padj_cutoff
    else:
        df["significant"] = (
            (df[padj_col] < padj_cutoff) &
            (df[logfc_col].abs() > logfc_cutoff)
        )
    
    plt.figure(figsize=(7, 5))
    
    # Non-significant genes
    plt.scatter(
        df.loc[~df["significant"], logfc_col],
        df.loc[~df["significant"], "minus_log10_padj"],
        s=8,
        alpha=0.4,
        label="Not significant"
    )
    
    # Significant genes
    plt.scatter(
        df.loc[df["significant"], logfc_col],
        df.loc[df["significant"], "minus_log10_padj"],
        s=12,
        alpha=0.8,
        label="Significant"
    )
    
    # Cutoff lines
    plt.axhline(-np.log10(padj_cutoff), linestyle="--", linewidth=1)
    
    if logfc_cutoff is not None:
        plt.axvline(logfc_cutoff, linestyle="--", linewidth=1)
        plt.axvline(-logfc_cutoff, linestyle="--", linewidth=1)
    
    # Label top significant genes
    top_genes = df[df["significant"]].sort_values(padj_col).head(label_top)
    
    for _, row in top_genes.iterrows():
        plt.text(
            row[logfc_col],
            row["minus_log10_padj"],
            str(row[gene_col]),
            fontsize=8)
    
    
    plt.xlabel("log2 fold change")
    plt.ylabel("-log10 adjusted p-value")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Data preparation

### Data loading

In [ ]:
file_path_FSHS1_1 = r"./data/GSM3487556_FSHD1.1.txt"
file_path_FSHS1_2 = r"./data/GSM3487557_FSHD1.2.txt"
file_path_FSHS2_1 = r"./data/GSM3487558_FSHD2.1.txt"
file_path_FSHS2_2 = r"./data/GSM3487559_FSHD2.2.txt"
file_path_ctrl1 = r"./data/GSM3487560_CTRL.1.txt"
file_path_ctrl2 = r"./data/GSM3487561_CTRL.2.txt"
file_path_DUX4_geneset = r"./data/DUX4_67_geneset.xlsx"
file_path_annotation = r"./data/annotation.gff3"

In [ ]:
RNA_sequencing_data = [
    {"sample_name":"FSHD1.1",
     "file_path": file_path_FSHS1_1},
    {"sample_name":"FSHD1.2",
     "file_path": file_path_FSHS1_2},
    {"sample_name":"FSHD2.1",
     "file_path": file_path_FSHS2_1},
    {"sample_name":"FSHD2.2",
     "file_path": file_path_FSHS2_2},
    {"sample_name":"CTRL.1",
     "file_path": file_path_ctrl1},
    {"sample_name":"CTRL.2",
     "file_path": file_path_ctrl2},
]

### Create adata and merge samples

In [ ]:
adata_list = []
sample_names = []
for my_sample_RNA_seq in RNA_sequencing_data:
    adata_list = adata_list + [read_count_matrix(my_sample_RNA_seq["file_path"], my_sample_RNA_seq["sample_name"])]
    sample_names = sample_names + [my_sample_RNA_seq["sample_name"]]

In [ ]:
# merge all samples into one anndata object, with sample names as batch labels
adata = ad.concat(
    adata_list,
    label = "sample_batch",
    keys = sample_names,
    join = "outer",
    index_unique = "-",
    fill_value=0
)

print(adata)
adata.write_h5ad("./data/combined_raw_counts.h5ad")



## QC

### Violin diagrams

In [ ]:
# load the combined anndata object
adata = sc.read_h5ad("./data/combined_raw_counts.h5ad")

# ensure cell names / gene names are unique
adata.obs_names_make_unique()
adata.var_names_make_unique()

sc.pp.calculate_qc_metrics(
    adata,
    inplace=True
)


adata.obs.groupby("sample_batch")[["total_counts", "n_genes_by_counts"]].describe()
sc.pl.violin(
    adata,
    ["total_counts", "n_genes_by_counts"],
    groupby="sample_batch",
    jitter=0.4,
    multi_panel=True
)

We produced 2 graphs: (1) the total number of counts per sample, (2) the total of expressed genes by counts.
Observations: 
(1) there seems to be signifant differences of total counts between samples (e.g. the average count is significantly different between FSHD1.1 and CTRL.2). The violin plots are not testing how significant that is. The conclusion is that there seems to be a significant effect of the batch. **That means that we should correct for the batch effect.**
(2) same idea for 2. The conclusion is that there seems to be a significant effect of the total of number of expressed genes by counts. **The total number of expressed genes by count should be also accounted for in the model specification.**

Using the violin plot observations we decided to calculate the factors that affect the expression measured: the total counts and number of genes per cell. We evaluate the different quantiles for each factor.

In [ ]:
# Summarize QC metrics by sample
qc_summary = (
    adata.obs
    .groupby("sample_batch", observed=True)[["total_counts", "n_genes_by_counts"]]
    .quantile([0.01, 0.05,0.1,0.25, 0.50, 0.75,0.9, 0.95, 0.99])
)

qc_summary

The total expression is variable between batches because the total counts we observe while sequencing is dependent on the amplification that was performed for each sample. It seems that the amplification was much smaller for CTRL2 than for the remaining samples. If we take a QC filtering common for all samples, that may create significantly unbalanced classes of deseases that will affect downstream the statistics. We propose to filter the cells that are below 5% of the total counts for at least one of the two factors.

### Correlation between number of genes and counts per cell

In [ ]:
# correlation between number of genes and  counts
samples = adata.obs["sample_batch"].unique()
# correlation between number of genes and total counts
plt.figure(figsize=(7, 5))

for sample in samples:
    sub = adata.obs[adata.obs["sample_batch"] == sample]
    plt.scatter(
        sub["total_counts"],
        sub["n_genes_by_counts"],
        s=5,
        alpha=0.4,
        label=sample
    )

plt.xlabel("Total counts per cell")
plt.ylabel("Number of genes per cell")
plt.legend(markerscale=3, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

corr_by_sample = (
    adata.obs
    .groupby("sample_batch", observed=True)
    .apply(lambda x: x["total_counts"].corr(x["n_genes_by_counts"], method="spearman"))
)

corr_by_sample

It seems that we have cell outliers in FSHD1.1 sample. Those are likely under-sequenced cells that we need to remove. 

### QC implementation

Based of the observations on the factors distributions, let's implement a filtering to make sure we keep only good quality sequenced cells. In the publication associated with the data, the authors did not tell what filtering criteria was used. Therefore, we cannot reproduce this step to replicate the data analysis of the publication. The only information found was in Figure S2 but the actual numbers used for filteing were not reported. This demonstrate how important the full description of the parameters choices made during the analysis pipelines is important to share and follow Findable Accessible Interoperable and Reproducible (FAIR) principles.
For the follow up analysis, we parametrized the filtering criteria to match the total cumber of filtered cells as reported in the publication.

In [ ]:
# Make a copy before filtering
adata_qc = adata.copy()

# Calculate QC metrics if not already calculated
sc.pp.calculate_qc_metrics(adata_qc, inplace=True)

# lower thresholds: Remove cells with very low RNA capture
min_genes = 700
min_counts = 2200

adata_qc = adata_qc[
    (adata_qc.obs["n_genes_by_counts"] > min_genes) &
    (adata_qc.obs["total_counts"] > min_counts),
    :
].copy()


print("Before filtering:", adata.n_obs, "cells")
print("After lower filtering:", adata_qc.n_obs, "cells")

In [ ]:
# second QC: remove high-count  cells

# Calculate counts per detected gene for each cell
adata_qc.obs["counts_per_gene"] = (
    adata_qc.obs["total_counts"] / 
    adata_qc.obs["n_genes_by_counts"]
)

print(adata_qc.obs["counts_per_gene"])

# Mark cells with high counts but low-genes
adata_qc.obs["qc_high_counts_per_gene"] = (
    adata_qc.obs["counts_per_gene"] > 8
)

# Count marked cells per sample
pd.crosstab(
    adata_qc.obs["sample_batch"],
    adata_qc.obs["qc_high_counts_per_gene"]
)

In [ ]:
# second QC
# filter: Remove cells with high counts per gene ratio
adata_qc2 = adata_qc[
    ~adata_qc.obs["qc_high_counts_per_gene"]
].copy()

print("Before second filtering:", adata_qc.n_obs, "cells")
print("After second filtering:", adata_qc2.n_obs, "cells")

In [ ]:
# correlation between number of genes and  counts
samples = adata_qc2.obs["sample_batch"].unique()
# correlation between number of genes and total counts
plt.figure(figsize=(7, 5))

for sample in samples:
    sub = adata_qc2.obs[adata_qc2.obs["sample_batch"] == sample]
    plt.scatter(
        sub["total_counts"],
        sub["n_genes_by_counts"],
        s=5,
        alpha=0.4,
        label=sample
    )

plt.xlabel("Total counts per cell")
plt.ylabel("Number of genes per cell")
plt.legend(markerscale=3, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

corr_by_sample = (
    adata_qc2.obs
    .groupby("sample_batch", observed=True)
    .apply(lambda x: x["total_counts"].corr(x["n_genes_by_counts"], method="spearman"))
)

corr_by_sample

# counts/genes correlation per cell after second qc

In [ ]:
# counts/genes correlation per cell after second qc
samples = adata_qc2.obs["sample_batch"].unique()
# correlation between number of genes and total counts
plt.figure(figsize=(7, 5))

for sample in samples:
    sub = adata_qc2.obs[adata_qc2.obs["sample_batch"] == sample]
    plt.scatter(
        sub["total_counts"],
        sub["n_genes_by_counts"],
        s=5,
        alpha=0.4,
        label=sample
    )

plt.xlabel("Total counts per cell")
plt.ylabel("Number of genes per cell")
plt.legend(markerscale=3, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

corr_by_sample = (
    adata_qc2.obs
    .groupby("sample_batch", observed=True)
    .apply(lambda x: x["total_counts"].corr(x["n_genes_by_counts"], method="spearman"))
)

corr_by_sample

# normalization

In [ ]:
# copy for normalization
adata_norm = adata_qc2.copy()

# Store raw counts before normalization
adata_norm.layers["counts"] = adata_norm.X.copy()

# Normalize each cell to the same total count
sc.pp.normalize_total(
    adata_norm,
    target_sum=1e4
)

# Log-transform the normalized data
sc.pp.log1p(adata_norm)

# Store log-normalized data
adata_norm.layers["lognorm"] = adata_norm.X.copy()

adata_norm.write_h5ad("combined_qc_normalized.h5ad")



# load DUX4-67 gene set

In [ ]:
# Add Ensembl ID column from adata gene names
adata_norm.var["ensembl_id"] = (
    adata_norm.var_names
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
    .str.replace(r"\.\d+$", "", regex=True)
)



In [ ]:
# Read file: DUX4 67 gene set
dux4_df = pd.read_excel(file_path_DUX4_geneset,header=1,engine="openpyxl")

# Extract Ensembl id from the DUX4-67 gene set
dux4_67_ensg = (
    dux4_df["ENSEMBL"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.replace(r"\.\d+$", "", regex=True)  # remove version number if present
    .unique()
    .tolist()
)

# Find DUX4-67 genes present in target dataset
dux4_genes_present = [
    gene for gene in dux4_67_ensg
    if gene in set(adata_norm.var["ensembl_id"])
]

print("DUX4-67 genes in file:", len(dux4_67_ensg))
print("DUX4-67 genes found in adata:", len(dux4_genes_present))


# cell classification

In [ ]:
#caculate the number of biomakers expresseed by cells
X_counts = adata_norm.layers["counts"]

# Get gene indices of DUX4-67 genes
dux4_gene_idx = np.where(
    adata_norm.var["ensembl_id"].isin(dux4_genes_present)
)[0]

# Extract raw counts of DUX4-67 genes
dux4_counts = X_counts[:, dux4_gene_idx]

# Count how many DUX4 biomarkers are expressed in each cell
if sparse.issparse(dux4_counts):
    adata_norm.obs["n_DUX4_targets_expressed"] = np.asarray(
        (dux4_counts > 0).sum(axis=1)
    ).ravel()
else:
    adata_norm.obs["n_DUX4_targets_expressed"] = (dux4_counts > 0).sum(axis=1)

adata_norm.obs["n_DUX4_targets_expressed"].describe()

In [ ]:
# Define DUX4-affected cells
adata_norm.obs["DUX4_affected"] = adata_norm.obs["n_DUX4_targets_expressed"] >= 5
adata_norm.obs["DUX4_affected"].value_counts()

# caculate counts of MYH3
myh3_ensg = "ENSG00000109063"

myh3_idx = np.where(adata_norm.var["ensembl_id"] == myh3_ensg)[0]

if len(myh3_idx) == 0:
    raise ValueError("MYH3 Ensembl ID was not found in adata.")

myh3_counts = X_counts[:, myh3_idx[0]]

if sparse.issparse(myh3_counts):
    myh3_counts = np.asarray(myh3_counts.todense()).ravel()
else:
    myh3_counts = np.asarray(myh3_counts).ravel()

adata_norm.obs["MYH3_counts"] = myh3_counts



In [ ]:
# > 1 read 
if sparse.issparse(dux4_counts):
    adata_norm.obs["n_DUX4_targets_expressed"] = np.asarray(
        (dux4_counts > 0).sum(axis=1)
    ).ravel()
else:
    adata_norm.obs["n_DUX4_targets_expressed"] = (
        dux4_counts > 0
    ).sum(axis=1)


# Initialize all cells as Other
adata_norm.obs["cell_classification"] = "Other"

# DUX4-affected cells:
# Cells expressing at least 5 DUX4-67 biomarkers
# Each biomarker is counted as expressed if raw count >= 1 read
adata_norm.obs.loc[
    adata_norm.obs["n_DUX4_targets_expressed"] >= 5,
    "cell_classification"
] = "DUX4-affected"

# Non-affected late myocytes:
# Cells with MYH3 raw counts >= 5
# and fewer than 5 expressed DUX4-67 biomarkers
adata_norm.obs.loc[
    (adata_norm.obs["MYH3_counts"] >= 5) &
    (adata_norm.obs["n_DUX4_targets_expressed"] < 5),
    "cell_classification"
] = "Non-affected late myocytes"


# Total number of cells in the current AnnData object
print("Total cell count:", adata_norm.n_obs)

# Count cells in each classification group
print(adata_norm.obs["cell_classification"].value_counts())


In [ ]:
# Check cell classification numbers
classification_summary = pd.crosstab(
    adata_norm.obs["sample_batch"],
    adata_norm.obs["cell_classification"]
)

classification_summary

# combat correction

In [ ]:
# Make a copy for ComBat analysis
adata_combat = adata_norm.copy()

# Use log-normalized data
adata_combat.X = adata_combat.layers["lognorm"].copy()

# Optional: remove genes expressed in very few cells before ComBat
sc.pp.filter_genes(adata_combat, min_cells=1)

# Run ComBat
sc.pp.combat(
    adata_combat,
    key="sample_batch"
)

# Remove genes with NaN after ComBat
adata_combat = remove_nan_genes(adata_combat)

# Save ComBat-corrected data
adata_combat.layers["combat"] = adata_combat.X.copy()


# PCA diagram (before combat and after combat)

In [ ]:
# select HVG （only DUX4 affected and non-affected cells)

# Subset DUX4-affected cells and non-affected late myocytes
adata_subset_before = adata_norm[
    adata_norm.obs["cell_classification"].isin([
        "DUX4-affected",
        "Non-affected late myocytes"
    ])
].copy()

# Use log-normalized data for HVG selection
adata_subset_before.X = adata_subset_before.layers["lognorm"].copy()

# Select highly variable genes on log-normalized data
sc.pp.highly_variable_genes(
    adata_subset_before,
    n_top_genes=2000,
    flavor="seurat"
)

# Save the same HVGs for before and after ComBat PCA
hvg_genes_subset = adata_subset_before.var_names[
    adata_subset_before.var["highly_variable"]
]
  
# PCA before ComBat correction
adata_pca_subset_before = adata_subset_before[:, hvg_genes_subset].copy()

# Use log-normalized data
adata_pca_subset_before.X = adata_pca_subset_before.layers["lognorm"].copy()

# Scale and run PCA
sc.pp.scale(adata_pca_subset_before, max_value=10)
sc.tl.pca(adata_pca_subset_before, svd_solver="arpack")

# PCA colored by cell classification
sc.pl.pca(
    adata_pca_subset_before,
    color="cell_classification",
    title="Subset PCA before ComBat: cell classification"
)

# PCA colored by sample
sc.pl.pca(
    adata_pca_subset_before,
    color="sample_batch",
    title="Subset PCA before ComBat: sample batch"
)

In [ ]:
# PCA after combat(using same HVG as before)

# Subset the ComBat-corrected object
adata_pca_subset_after = adata_combat[
    adata_combat.obs["cell_classification"].isin([
        "DUX4-affected",
        "Non-affected late myocytes"
    ])
].copy()

# Use the same HVGs as before
adata_pca_subset_after = adata_pca_subset_after[:, hvg_genes_subset].copy()

# Use ComBat-corrected data
adata_pca_subset_after.X = adata_pca_subset_after.layers["combat"].copy()

# Scale and run PCA
sc.pp.scale(adata_pca_subset_after, max_value=10)
sc.tl.pca(adata_pca_subset_after, svd_solver="arpack")

# PCA colored by cell classification
sc.pl.pca(
    adata_pca_subset_after,
    color="cell_classification",
    title="Subset PCA after ComBat: cell classification"
)

# PCA colored by sample
sc.pl.pca(
    adata_pca_subset_after,
    color="sample_batch",
    title="Subset PCA after ComBat: sample batch"
)

# optional: PCA before/after combat correction (other cells included)

In [ ]:
# PCA before batch correction (other cells included)
adata_pca_before = adata_combat.copy()

# Use log-normalized data
adata_pca_before.X = adata_pca_before.layers["lognorm"].copy()

# Select highly variable genes
sc.pp.highly_variable_genes(
    adata_pca_before,
    n_top_genes=2000,
    flavor="seurat"
)

# Keep only highly variable genes
adata_pca_before = adata_pca_before[:, adata_pca_before.var["highly_variable"]].copy()

# Scale data before PCA
sc.pp.scale(
    adata_pca_before,
    max_value=10
)

# Run PCA
sc.tl.pca(
    adata_pca_before,
    svd_solver="arpack"
)

# Plot PCA colored by sample
sc.pl.pca(
    adata_pca_before,
    color="sample_batch",
    title="PCA before batch correction"
)

# Plot PCA colored by cell classification
sc.pl.pca(
    adata_pca_before,
    color="cell_classification",
    title="PCA before batch correction"
)

In [ ]:
# PCA after ComBat correction(other cells included)
adata_pca_after = adata_combat.copy()

# Use ComBat-corrected data
adata_pca_after.X = adata_pca_after.layers["combat"].copy()

# Use the same highly variable genes as before
hvg_genes = adata_pca_before.var_names

adata_pca_after = adata_pca_after[:, hvg_genes].copy()

# Scale data before PCA
sc.pp.scale(
    adata_pca_after,
    max_value=10
)

# Run PCA
sc.tl.pca(
    adata_pca_after,
    svd_solver="arpack"
)

# Plot PCA colored by sample
sc.pl.pca(
    adata_pca_after,
    color="sample_batch",
    title="PCA after ComBat correction"
)

# Plot PCA colored by cell classification
sc.pl.pca(
    adata_pca_after,
    color="cell_classification",
    title="PCA after ComBat correction"
)

# differential analysis using disease group

In [ ]:
# Use batch effect removed data for differential expression
adata_de = adata_combat.copy()

# Use ComBat-corrected data
adata_de.X = adata_de.layers["combat"].copy()


In [ ]:
# Define disease_group = FSHD1 / FSHD2 / Control
group_map = {
    "FSHD1.1": "FSHD1",
    "FSHD1.2": "FSHD1",
    "FSHD2.1": "FSHD2",
    "FSHD2.2": "FSHD2",
    "CTR1": "Control",
    "CTR2": "Control"}

adata_de.obs["disease_group"] = (
    adata_de.obs["sample_batch"]
    .astype(str)
    .map(group_map))

In [ ]:
# DE results
disease_groups = ["FSHD1", "FSHD2"]

de_results = []

for disease in disease_groups:
    result = de_disease_group(
        adata_de,
        disease_name= disease,
        de_layer="combat"
    )
    
    if result is not None:
        de_results.append(result)

de_by_disease = pd.concat(de_results, ignore_index=True)

In [ ]:
for disease in disease_groups:
    sub = de_by_disease[de_by_disease["disease_group"] == disease]
    print(disease)
    print("raw p < 0.05:", (sub["pvals"] < 0.05).sum())
    print("adj p < 0.05:", (sub["pvals_adj"] < 0.05).sum())

In [ ]:
# Create clean gene ID column
de_by_disease["gene_id_clean"] = (
    de_by_disease["gene_id"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.\d+$", "", regex=True)
)

In [ ]:
# Create significant gene sets for each disease group
#P > 0.05 and logFC > 1
sig_gene_sets = {}

for disease in ["FSHD1", "FSHD2"]:
    genes = set(
        de_by_disease.loc[
            (de_by_disease["disease_group"] == disease) &
            (de_by_disease["pvals_adj"] < 0.05) &
            (de_by_disease["logfoldchanges"] > 1),
            "gene_id_clean"
        ].dropna()
    )
    
    sig_gene_sets[disease] = genes

for disease, genes in sig_gene_sets.items():
    print(disease, len(genes))

In [ ]:
# Define disease-specific and common genes
fshd1_specific = sig_gene_sets["FSHD1"] - sig_gene_sets["FSHD2"]
fshd2_specific = sig_gene_sets["FSHD2"] - sig_gene_sets["FSHD1"]
common_genes = sig_gene_sets["FSHD1"] & sig_gene_sets["FSHD2"]

print("FSHD1 significant:", len(sig_gene_sets["FSHD1"]))
print("FSHD2 significant:", len(sig_gene_sets["FSHD2"]))
print("FSHD1-specific:", len(fshd1_specific))
print("FSHD2-specific:", len(fshd2_specific))
print("Common:", len(common_genes))

In [ ]:
# Annotate FSHD1-specific and FSHD2-specific gene sets
fshd1_specific_df = annotate_gene_set(
    fshd1_specific,
    id_to_symbol,
    "FSHD1 -specific")

fshd2_specific_df = annotate_gene_set(
    fshd2_specific,
    id_to_symbol,
    "FSHD2 -specific")

common_genes_df = annotate_gene_set(
    common_genes,
    id_to_symbol,
    "Common")

In [ ]:
# Save annotated gene sets
fshd1_specific_df.to_csv(
    "FSHD1_specific_reconstructed_with_gene_symbols.csv",
    index=False)

fshd2_specific_df.to_csv(
    "FSHD2_specific_reconstructed_with_gene_symbols.csv",
    index=False)

common_genes_df.to_csv(
    "Common_reconstructed_with_gene_symbols.csv",
    index=False)

# optional: differential analysis using sample batch

In [ ]:
# differential analysis using sample batch

# Define FSHD samples
fshd_samples = ["FSHD1.1", "FSHD1.2", "FSHD2.1", "FSHD2.2"]


de_results = []

for sample in fshd_samples:
    result = run_de_for_sample(adata_de, sample)
    if result is not None:
        de_results.append(result)

de_per_sample = pd.concat(de_results, ignore_index=True)


In [ ]:
for sample in fshd_samples:
    sub = de_per_sample[de_per_sample["sample"] == sample]
    print(sample)
    print("raw p < 0.05:", (sub["pvals"] < 0.05).sum())
    print("adj p < 0.05:", (sub["pvals_adj"] < 0.05).sum())

In [ ]:
#Wilcoxon Rank Sum Test; BH FDR-corrected P-value < 0.05
# Keep significant differentially expressed genes

de_per_sample_sig = de_per_sample[
    (de_per_sample["pvals_adj"] < 0.05) &
    (de_per_sample["logfoldchanges"].abs() > 1)
].copy()





In [ ]:
# Create significant gene sets for each sample
sig_gene_sets = {}

for sample in fshd_samples:
    genes = set(
        de_per_sample_sig.loc[
            de_per_sample_sig["sample"] == sample,
            "gene_id"
        ]
    )
    sig_gene_sets[sample] = genes

# Check number of significant genes per sample
for sample, genes in sig_gene_sets.items():
    print(sample, len(genes))

In [ ]:
# Genes significant in FSHD1 samples but not in FSHD2 samples
fshd1_specific = (
    sig_gene_sets["FSHD1.1"] |
    sig_gene_sets["FSHD1.2"]
) - (
    sig_gene_sets["FSHD2.1"] |
    sig_gene_sets["FSHD2.2"]
)

print("FSHD1-specific genes:", len(fshd1_specific))

In [ ]:
# Genes significant in FSHD2 samples but not in FSHD1 samples
fshd2_specific = (
    sig_gene_sets["FSHD2.1"] |
    sig_gene_sets["FSHD2.2"]
) - (
    sig_gene_sets["FSHD1.1"] |
    sig_gene_sets["FSHD1.2"]
)

print("FSHD2-specific genes:", len(fshd2_specific))

In [ ]:
# Annotate FSHD1-specific and FSHD2-specific gene sets
fshd1_specific_df = annotate_gene_set(
    fshd1_specific,
    id_to_symbol,
    "FSHD1-specific"
)

fshd2_specific_df = annotate_gene_set(
    fshd2_specific,
    id_to_symbol,
    "FSHD2-specific"
)

print(fshd1_specific_df)
print(fshd2_specific_df)

In [ ]:
# Save annotated gene sets
fshd1_specific_df.to_csv(
    "FSHD1_specific_reconstructed_with_gene_symbols.csv",
    index=False
)

fshd2_specific_df.to_csv(
    "FSHD2_specific_reconstructed_with_gene_symbols.csv",
    index=False
)

# volcano plot

In [ ]:
adata_de.X = adata_de.layers["combat"].copy()

# transfer gene_id to gene_sympol

# Add gene symbols to DE result
de_by_disease["gene_id_clean"] = (
    de_by_disease["gene_id"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.\d+$", "", regex=True)
)

de_by_disease["gene_symbol"] = de_by_disease["gene_id_clean"].map(id_to_symbol)

# Use gene_id as backup if gene symbol is missing
de_by_disease["gene_label"] = de_by_disease["gene_symbol"].fillna(
    de_by_disease["gene_id"]
)

# Add gene symbols to adata.var
adata_de.var["ensembl_id_clean"] = (
    adata_de.var["ensembl_id"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.\d+$", "", regex=True)
)

adata_de.var["gene_symbol"] = adata_de.var["ensembl_id_clean"].map(id_to_symbol)
adata_de.var["gene_label"] = adata_de.var["gene_symbol"].fillna(
    adata_de.var["ensembl_id"]
)

In [ ]:
# volcano plots for each sample

for disease in disease_groups:
    sub = de_by_disease[
        de_by_disease["disease_group"] == disease
    ].copy()
    
    # number of significant genes
    print(disease, "DE genes:", sub.shape[0])
    print("Significant genes:", ((sub["pvals_adj"] < 0.05) & (sub["logfoldchanges"].abs() > 1)).sum())
    
    if sub.shape[0] == 0:
        print(disease, "has no DE result, skipped.")
        continue
    
    #plot
    plot_volcano(
        sub,
        title=f"{disease}: DUX4-affected vs non-affected late myocytes",
        gene_col="gene_symbol",
        padj_cutoff=0.05,
        logfc_cutoff=1)

# heat map

In [ ]:
top_gene_summary = []
top_genes = []
top_genes_by_disease = {}

for disease in disease_groups: 
    sub = de_by_disease[
        (de_by_disease["disease_group"] == disease) &
        (de_by_disease["pvals_adj"] < 0.05) &
        (de_by_disease["logfoldchanges"] > 1)
    ].copy()
    
    sub = sub.sort_values("pvals_adj")
    
    # Keep top 10 genes
    selected = sub["gene_id_clean"].head(10).tolist()
    # Remove duplicated genes
    top_genes = list(dict.fromkeys(top_genes))

    top_genes_by_disease[disease] = selected
    top_genes.extend(selected)
    
    top_gene_summary.append({
        "disease": disease,
        "top_genes_selected": len(selected)
    })

top_gene_summary_df = pd.DataFrame(top_gene_summary)

print("Number of genes for heatmap:", len(top_genes))
print(top_gene_summary_df)

In [ ]:
# Build mapping: clean Ensembl ID -> adata var_name
id_to_varname = dict(zip(
    adata_de.var["ensembl_id_clean"],
    adata_de.var_names
))

# Convert selected gene IDs to var_names used by adata
top_var_names = [
    id_to_varname[g]
    for g in top_genes
    if g in id_to_varname
]



In [ ]:
# Keep FSHD samples and two cell classes
adata_heat = adata_de[
    (adata_de.obs["disease_group"].isin([
        "FSHD1", "FSHD2"
    ])) &
    (adata_de.obs["cell_classification"].isin([
        "DUX4-affected",
        "Non-affected late myocytes"
    ]))
].copy()

# Use combat-corrected data
adata_heat.X = adata_heat.layers["combat"].copy()

# Create sample + cell class group
adata_heat.obs["sample_cell_class"] = (
    adata_heat.obs["sample_batch"].astype(str)
    + " | "
    + adata_heat.obs["cell_classification"].astype(str)
)

In [ ]:
# Create short sample labels
adata_heat.obs["sample_short"] = (
    adata_heat.obs["sample_batch"]
    .astype(str)
    .str.replace("FSHD", "", regex=False)
)

# Create short class labels
class_map = {
    "DUX4-affected": "A",
    "Non-affected late myocytes": "B"
}

adata_heat.obs["class_short"] = (
    adata_heat.obs["cell_classification"]
    .map(class_map)
)

# Create heatmap column labels
adata_heat.obs["heat_group"] = (
    adata_heat.obs["sample_short"].astype(str)
    + "-"
    + adata_heat.obs["class_short"].astype(str)
)

# Define column order
group_order = [
    "1.1-A", "1.1-B",
    "1.2-A", "1.2-B",
    "2.1-A", "2.1-B",
    "2.2-A", "2.2-B"
]

# Keep only groups that exist
group_order = [
    g for g in group_order
    if g in adata_heat.obs["heat_group"].unique()
]

adata_heat.obs["heat_group"] = pd.Categorical(
    adata_heat.obs["heat_group"],
    categories=group_order,
    ordered=True
)

print(adata_heat.obs["heat_group"].value_counts())

In [ ]:
# Extract combat version expression of selected genes
X = adata_heat[:, top_var_names].layers["combat"]

if sparse.issparse(X):
    X = X.toarray()

# Create expression dataframe
expr_df = pd.DataFrame(
    X,
    index=adata_heat.obs_names,
    columns=top_var_names
)

# Add group information
expr_df["heat_group"] = adata_heat.obs["heat_group"].values

# Calculate average expression per group
mean_expr = (
    expr_df
    .groupby("heat_group", observed=True)
    .mean()
    .T
)

# Reorder columns
mean_expr = mean_expr[group_order]


In [ ]:
# Z-score scaling per gene
heat_data = mean_expr.sub(mean_expr.mean(axis=1), axis=0)
heat_data = heat_data.div(mean_expr.std(axis=1).replace(0, np.nan), axis=0)

# Replace NaN caused by zero standard deviation
heat_data = heat_data.fillna(0)

# Limit extreme values for clearer colors
heat_data = heat_data.clip(-1.5, 1.5)

In [ ]:
# Add gene symbol labels
if "gene_symbol" not in adata_heat.var.columns:
    adata_heat.var["ensembl_id_clean"] = (
        adata_heat.var_names
        .astype(str)
        .str.replace('"', '', regex=False)
        .str.strip()
        .str.replace(r"\.\d+$", "", regex=True)
    )
    adata_heat.var["gene_symbol"] = (
        adata_heat.var["ensembl_id_clean"].map(id_to_symbol)
    )

# Create label map
gene_label_map = (
    adata_heat.var["gene_symbol"]
    .fillna(pd.Series(adata_heat.var_names, index=adata_heat.var_names))
    .to_dict()
)

# Replace ENSG index by gene symbols
gene_labels = [
    gene_label_map.get(g, g)
    for g in heat_data.index
]

heat_data.index = gene_labels

In [ ]:
# Plot heatmap: genes as rows, samples/groups as columns
fig_height = max(5, 0.35 * heat_data.shape[0])
fig_width = max(7, 0.7 * heat_data.shape[1])

plt.figure(figsize=(fig_width, fig_height))

plt.imshow(
    heat_data.values,
    aspect="auto",
    cmap="bwr",
    vmin=-1.5,
    vmax=1.5
)

plt.colorbar(label="Scaled expression")

plt.xticks(
    ticks=np.arange(heat_data.shape[1]),
    labels=heat_data.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=np.arange(heat_data.shape[0]),
    labels=heat_data.index
)

plt.xlabel("Sample group(A for DUX4-effected sample, B for non-affected late myocytes)")
plt.ylabel("Gene")
plt.title("Top expressed genes from per-sample analysis")

plt.tight_layout()
plt.show()